# Fraud Model Training

This notebook trains a fraud detection model on the sampled, processed dataset and evaluates the result on a held-out validation split.


In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent

processed_path = project_root / "data" / "processed" / "fraud_sampled.csv"
print(f"Processed file: {processed_path}")

df = pd.read_csv(processed_path)
print(df.head())
print(f"Shape: {df.shape}")
print(df["isFraud"].value_counts(normalize=True).to_dict())


Processed file: C:\Users\MY PC\Naresh IT\AI Projects store_fraud_detection\AI Projects store_fraud_detection\data\processed\fraud_sampled.csv
   step      type     amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0    15   PAYMENT   14183.02  C1511752447      424164.19       409981.17   
1    13   CASH_IN  134116.02   C624343034    24950988.41     25085104.43   
2   323  CASH_OUT  264977.25   C926806709       20303.00            0.00   
3   349   PAYMENT   16723.10   C543711584       20373.00         3649.90   
4   369  CASH_OUT  216305.21  C1790143581           0.00            0.00   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979774744            0.00            0.00        0               0  
1  C1113941243       608933.74       474817.72        0               0  
2  C1224322366       991497.14      1256474.39        0               0  
3  M2102407654            0.00            0.00        0               0  
4  C1323576474      3180972.31 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

feature_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "type",
]

X = pd.get_dummies(df[feature_columns], columns=["type"], drop_first=True)
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds, target_names=["Non-Fraud", "Fraud"]))
print(confusion_matrix(y_test, preds))


              precision    recall  f1-score   support

   Non-Fraud       1.00      0.95      0.97     32852
       Fraud       0.47      0.92      0.63      1643

    accuracy                           0.95     34495
   macro avg       0.74      0.94      0.80     34495
weighted avg       0.97      0.95      0.96     34495

[[31175  1677]
 [  126  1517]]


In [3]:
model_path = project_root / "models" / "production" / "fraud_model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)

import joblib
joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")


Model saved to: C:\Users\MY PC\Naresh IT\AI Projects store_fraud_detection\AI Projects store_fraud_detection\models\production\fraud_model.joblib
